In [ ]:
import mlx.core as mx
from mlx_lm import load
import pandas as pd

messages = [
    {
        "role": "user",
        "content": """Write a Python function `clamp(x, low, high)` that returns `x` restricted to the inclusive range `[low, high]`.

Requirements:
- If `x` is less than `low`, return `low`.
- If `x` is greater than `high`, return `high`.
- Otherwise return `x`.
- Reply with only the function.""",
    },
    {
        "role": "assistant",
        "content": """```python
def clamp(x, low, high):
    if x < low:
        return high
    if x > high:
        return low
    return x
```""",
    },
    {
        "role": "user",
        "content": """Your submission failed the tests, so the issue remains unresolved.

clamp(-1, 0, 10) returned 10, expected 0.
clamp(11, 0, 10) returned 0, expected 10.
clamp(4, 0, 10) returned 4, expected 4.

You swapped the two bounds. Values below `low` must be brought up to `low`, and values above `high` must be brought down to `high`.

The entire environment, including all files and state, has been restored to its exact state at the beginning of the task. You will now receive the same task again. Start from scratch, applying what you learned from this attempt.

Write a Python function `clamp(x, low, high)` that returns `x` restricted to the inclusive range `[low, high]`.

Requirements:
- If `x` is less than `low`, return `low`.
- If `x` is greater than `high`, return `high`.
- Otherwise return `x`.
- Reply with only the function.""",
    },
    {
        "role": "assistant",
        "content": """```python
def clamp(x, low, high):
    if x < low:
        return high
    if x > high:
        return low
    return x
```""",
    },
]

model, tokenizer = load("mlx-community/Qwen2.5-Coder-7B-Instruct-4bit")

ids = mx.array(tokenizer.apply_chat_template(messages))
hidden = model.model(mx.concat([ids, ids])[None])[0,len(ids):]
logits = model.lm_head(hidden)
probs = mx.softmax(logits, axis=-1)
labels = ids[1:]
pred_ids = mx.argsort(probs, axis=-1)[:,:-4:-1]
mx.eval(pred_ids)

rows = []
for i in range(len(labels)):
    def cell(id):
        token = tokenizer.decode([id.tolist()], skip_special_tokens=False)
        return f"{token!r} @ {probs[i,id]:.3f}"
    
    row = {"label": cell(labels[i])}
    for j, pred_id in enumerate(pred_ids[i]):
        row[f"pred_{j}"] = cell(pred_id)
    rows.append(row)
df = pd.DataFrame(rows)
df.tail(32)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

,label,pred_0,pred_1,pred_2
376,"',' @ 1.000","',' @ 1.000",':' @ 0.000,' low' @ 0.000
377,' low' @ 1.000,' low' @ 1.000,' lo' @ 0.000,'low' @ 0.000
378,"',' @ 1.000","',' @ 1.000","',\n' @ 0.000","' ,' @ 0.000"
379,' high' @ 1.000,' high' @ 1.000,' max' @ 0.000,'.high' @ 0.000
380,'):\n' @ 1.000,'):\n' @ 1.000,'):' @ 0.000,'):\r\n' @ 0.000
381,' ' @ 1.000,' ' @ 1.000,' ' @ 0.000,' if' @ 0.000
382,' if' @ 0.907,' if' @ 0.907,' return' @ 0.093,"' """"""' @ 0.000"
383,' x' @ 1.000,' x' @ 1.000,' low' @ 0.000,' high' @ 0.000
384,' <' @ 0.996,' <' @ 0.996,' >' @ 0.003,' <=' @ 0.001
385,' low' @ 0.999,' low' @ 0.999,' high' @ 0.001,'low' @ 0.000


There's credit attribution, but it falls off after the first instance because the model's disagreement isn't reflected in the trajectory and so it tries to conform with what's already there, even if it's incorrect. I think this means we can't do a naive offline approach. Maybe we can try chunks?